In [ ]:
! pip install --upgrade -q pandas numpy scikit-learn imbalanced-learn shap

In [ ]:
from collections import Counter

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.ensemble import RandomForestClassifier

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline

import shap

from utils.constants import *

In [ ]:
df = pd.read_csv("../data/3_gold/dataset-processed-rf.csv")
X = df.drop("class", axis=1)
y = df["class"]
y = y.map(TARGET_LABEL_MAP)

feature_names = X.columns.tolist() 

test_indices = []
train_indices = []

for label in [0, 1, 2]:
    indices = y[y == label].index
    
    # Split indices: first 500 go to test, the rest go to train
    test_indices.extend(indices[:500])
    train_indices.extend(indices[500:])

X_test = X.loc[test_indices]
y_test = y.loc[test_indices]

X_train = X.loc[train_indices]
y_train = y.loc[train_indices]

print(f"Test set shape: {y_test.shape}")
print(f"Test class counts:\n{y_test.value_counts()}")
print(f"Train set shape: {y_train.shape}")

In [ ]:
params = {
    'n_estimators': 100, 
    'max_depth': 10,
    'class_weight': 'balanced',
    'random_state': RANDOM_STATE
}

In [ ]:
model = RandomForestClassifier(**params)
model.fit(X_train, y_train)

preds = model.predict(X_test)
f1 = f1_score(y_test, preds, average='macro')
print("Test F1-score:", f1)

# Use f-beta score with beta=2 for more emphasis on recall
from sklearn.metrics import fbeta_score

f2 = fbeta_score(y_test, preds, beta=2, average='macro')
print("Test F2-score:", f2)

from sklearn.metrics import classification_report

print("Classification Report:\n", classification_report(y_test, preds, target_names=TARGET_NAMES))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

disp = ConfusionMatrixDisplay.from_predictions(
    y_test, preds, 
    display_labels=TARGET_NAMES, cmap=plt.cm.Blues, normalize='true'
)
disp.im_.set_clim(0, 1)
plt.title("Confusion Matrix - Random Forest")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import shap

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

for i, class_name in enumerate(TARGET_NAMES):
    print(f"Generating SHAP summary for class: {class_name}")
    
    if isinstance(shap_values, list):
        class_shap_values = shap_values[i]
    else:
        class_shap_values = shap_values[:, :, i]

    plt.figure()
    shap.summary_plot(
        class_shap_values, 
        X_test, 
        feature_names=feature_names,
        show=False
    )
    
    plt.title(f"SHAP Summary: {class_name}")
    plt.tight_layout()
    plt.show()